In [ ]:
# Aman Kumar
from langchain.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env")

api_key = os.getenv("CO_API_KEY")
print(api_key)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#prompt templates
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)

In [ ]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

response = chat_model.invoke("What is Capital of USA?")
print(response.content)

In [ ]:
#LLM Chains
llm = ChatOpenAI(model="gpt-4o-mini")

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

learn_prompt = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Deep learning"

chain = LLMChain(llm=llm, prompt=learn_prompt)

result = chain.invoke({"AI_topic": description})
print(result["text"])


### What is a Tool in LangChain?

**Tool** is a function that the LangChain agent can **call to perform a specific task**.

For example:

```python
def my_tool_function(query: str) -> str:
    return f"Tool1 response: {query}"
````

You convert this Python function into a LangChain Tool:

```python
my_tool = Tool.from_function(
    func=my_tool_function,
    name="simple_tool",
    description="A simple tool"
)
```

The flow is:

```text
Python Function
      ↓
LangChain Tool
      ↓
Agent can call the Tool
```

In your code, the agent has two tools:

```text
Agent
  │
  ├── simple_tool  → my_tool_function()
  │
  └── simple_tool2 → my_tool_function2()
```

The important point is:

> The **LLM does not execute the Python function itself**.
> The LLM decides **which tool to call and what input to provide**, and LangChain executes the actual Python function.
> A **Tool** is an external capability or function that an agent can invoke when it needs to perform a specific task.


### Using Simple Function

In [ ]:
# Import the required classes
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool

# Define a custom tool function
def my_tool_function(query: str) -> str:
    # Simply returns the input query with a prefix
    return f"Tool1 response: {query}"

# Define a custom tool function
def my_tool_function2(query: str) -> str:
    # Simply returns the input query with a prefix
    return f"Tool response: {query}"

# Convert the Python function into a LangChain Tool
my_tool = Tool.from_function(
    func=my_tool_function,         # Function to execute
    name="simple_tool",            # Tool name used by the agent
    description="A simple tool"    # Description helps the agent decide when to use it
)

# Convert the Python function into a LangChain Tool
my_tool2 = Tool.from_function(
    func=my_tool_function2,         # Function to execute
    name="simple_tool2",            # Tool name used by the agent
    description="A simple tool2"    # Description helps the agent decide when to use it
)

# Create a Tavily search tool (requires TAVILY_API_KEY)
# tavily_search = TavilySearchResults(max_results=2)

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o-mini")

# List of tools available to the agent
# tools = [tavily_search, my_tool]   # Agent can use both Tavily Search and custom tool
tools = [my_tool, my_tool2]                    # Agent can use only the custom tools

# Create the agent
agent = initialize_agent(
    tools,                                       # Tools available to the agent
    llm,                                         # Language model
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # Uses ReAct reasoning based on tool descriptions
    verbose=True                                 # Shows the agent's reasoning steps
)

# Ask the agent a question
response = agent.run("What's the weather like today in London?")

# Print the final response
print(response)

Certainly. This output shows how a **ReAct (Reason + Act)** agent thinks step-by-step while trying to answer your question.

Let's go through each part.

**1. Agent starts**

```text
> Entering new AgentExecutor chain...
```

This indicates that the LangChain agent has started executing your request.

---

**2. First Thought**

```text
I need to find out the current weather conditions in London.
I will use the simple_tool to query the weather information.
```

This is the agent's reasoning.

It analyzes the question:

> "What's the weather like today in London?"

Since the only available tool is:

```python
tools = [my_tool]
```

and its description is:

```python
description="A simple tool"
```

the agent decides that this is the only tool it can use.

---

**3. Action**

```text
Action: simple_tool
```

The agent has decided to call your custom tool.

Internally, it is equivalent to:

```python
my_tool_function("current weather in London")
```

---

**4. Action Input**

```text
Action Input:
"current weather in London"
```

This is the string passed to your tool.

So your function receives:

```python
query = "current weather in London"
```

---

**5. Observation**

```text
Observation:
Tool response: current weather in London
```

This is the output returned by your tool.

Remember your function:

```python
def my_tool_function(query):
    return f"Tool response: {query}"
```

So it simply echoes the input.

It **does not**:

- Search Google
- Call a weather API
- Know London's temperature

It only returns:

```text
Tool response: current weather in London
```

---

**6. Second Thought**

```text
The tool provided a general response,
but I need specific weather details...
```

The agent reasons:

> "That wasn't useful."

Because it expected weather information but only got an echoed sentence.

---

**7. Another Action**

The agent tries again.

```text
Action:
simple_tool

Action Input:
What's the current temperature and weather conditions in London?
```

Again your function receives:

```python
my_tool_function(
    "What's the current temperature and weather conditions in London?"
)
```

Output:

```text
Tool response:
What's the current temperature and weather conditions in London?
```

Again, no real weather information.

---

**8. Loop Continues**

The agent keeps thinking:

```text
Maybe I should ask differently.
```

It tries different queries such as:

```text
London weather details today...
```

Then:

```text
Provide detailed weather information...
```

Then:

```text
Get current weather report...
```

Each time your function simply echoes the input instead of returning actual weather data.

---

**9. Final Thought**

Eventually the agent realizes:

```text
The tool is not providing specific weather details.
```

The ReAct agent concludes:

> "This tool cannot answer my question."

---

**10. Final Answer**

```text
I am unable to provide the current weather information...
```

The agent gives up because it has exhausted reasonable attempts.

---

**Why did it keep trying?**

Because **ReAct agents are designed to think and retry**.

The execution cycle is:

```text
Question
     │
     ▼
Thought
     │
     ▼
Choose Tool
     │
     ▼
Execute Tool
     │
     ▼
Observe Result
     │
     ▼
Enough?
 ┌───────┴────────┐
 │                │
No               Yes
 │                │
 ▼                ▼
Think Again     Final Answer
```

This retry behavior is one of the key features of the ReAct agent.

---

**Why couldn't it answer?**

Your tool is intentionally very simple:

```python
def my_tool_function(query):
    return f"Tool response: {query}"
```

No matter what is asked:

```text
"Weather?"
```

or

```text
"Temperature?"
```

or

```text
"Humidity?"
```

it only returns:

```text
Tool response: <input>
```

There is **no external data source**, so the tool cannot provide real-time weather information.

### Using DuckDuckGoSearchRun

DuckDuckGoSearchRun is a LangChain tool that lets an AI agent search the web using the DuckDuckGo search engine.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

dg_search = DuckDuckGoSearchRun()

tools = [dg_search, my_tool]                    # Agent can use both tools

# Create the agent
agent = initialize_agent(
    tools,                                       # Tools available to the agent
    llm,                                         # Language model
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # Uses ReAct reasoning based on tool descriptions
    verbose=True                                 # Shows the agent's reasoning steps
)

# Ask the agent a question
response = agent.run("What's the weather like today in London?")

# Print the final response
print(response)

### LLM + Tools

So your understanding is correct — `dg_search` gets used for retrieving information and `summarize_tool` for condensing it — but critically, __it's the LLM itself (via the ReAct prompting pattern), not your code, that decides the order and whether to call a tool at all.__ This is also why the `description` field matters so much: if it's vague or misleading, the LLM may pick the wrong tool or skip a tool it should have used. If you ever want deterministic control (e.g., always search then always summarize), you'd replace the agent with an explicit chain/sequence instead of using `initialize_agent`.

One note unrelated to your question: this notebook is using the older `LLMChain` and `initialize_agent` APIs, which are deprecated in favor of LangChain's newer `LCEL` (`prompt | llm`) and `create_react_agent`/`AgentExecutor` or LangGraph patterns — still fine for learning purposes, just worth knowing if you extend this later.


In [ ]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-4o-mini")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)

summarize_tool = Tool.from_function(
    func=llm_chain.run,
    name="Summarizer",
    description="Summarizes a web page"
)

In [ ]:
tools = [dg_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

In [ ]:
response = agent.invoke({"input": "Who invented the World Wide Web and what impact did it have?"})
print(response["output"])